# 使用预训练scGPT模型进行参考映射（Reference Mapping）

本教程演示如何使用scGPT模型将细胞嵌入并映射到参考嵌入。然后可以将参考细胞的元标签（如细胞类型或疾病状态）传播到查询细胞。我们特别使用`scGPT_human`模型直接提供嵌入。在这种零样本设置下，无需进一步训练。整个工作流程可以快速完成，并且我们观察到相当高的准确性。

本示例使用胰腺数据集。请从 https://drive.google.com/drive/folders/1s9XjcSiPC-FYV3VeHrEa7SeZetrthQVV?usp=sharing 下载。

我们提供以下两种参考映射模式：

1. **使用自定义参考数据集及其注释**。将查询集中的未知细胞映射到这个参考数据集。这说明了用户已经有类似样本的注释并希望快速将注释转移到新收集的样本的用例。

2. **使用我们之前从CellXGene收集的超过3300万个细胞作为参考**。将查询集中的未知细胞映射到这个参考图谱。这说明了用户希望将自己的数据映射到大型参考图谱的通用用例。例如，这可以是了解新收集样本的细胞组成的快速第一步。

根据您的用例，您可能只需要**应用两种模式中的一种**。

**注意**：参考映射是一项新的实验性功能。


In [1]:
# 导入必要的库
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mode
import scanpy as sc
import sklearn
import warnings

# 添加scGPT路径
sys.path.insert(0, "../")
import scgpt as scg

# 导入faiss用于快速相似度搜索（可选但推荐）
try:
    import faiss
    faiss_imported = True
    print("✅ faiss已安装，将使用快速相似度搜索")
except ImportError:
    faiss_imported = False
    print("⚠️ faiss未安装！强烈建议安装以获得快速相似度搜索。")
    print("安装方法：https://github.com/facebookresearch/faiss/wiki/Installing-Faiss")

# 忽略资源警告
warnings.filterwarnings("ignore", category=ResourceWarning)

## 方法一：使用自定义参考数据集进行参考映射

In [2]:
# 设置路径和参数
model_dir = Path("../save/scGPT_human")  # 预训练模型目录
adata = sc.read_h5ad("../data/annotation_pancreas/demo_train.h5ad")  # 参考数据
cell_type_key = "Celltype"  # 细胞类型列名
gene_col = "index"  # 基因名列名

In [3]:
# 使用scGPT获取参考数据的细胞嵌入
ref_embed_adata = scg.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=cell_type_key,  # 可选参数，用于保存元信息
    batch_size=64,
    return_new_adata=True,
)

# # 如果在CPU上运行（不推荐，因为速度较慢）
# ref_embed_adata = scg.tasks.embed_data(
#     adata,
#     model_dir,
#     gene_col=gene_col,
#     obs_to_save=cell_type_key,
#     batch_size=64,
#     device="cpu",
#     use_fast_transformer=False,
#     return_new_adata=True,
# )

# 可选步骤：使用嵌入可视化参考数据集
sc.pp.neighbors(ref_embed_adata, use_rep="X")  # 构建邻居图
sc.tl.umap(ref_embed_adata)  # 降维到UMAP
sc.pl.umap(ref_embed_adata, color=cell_type_key, frameon=False, wspace=0.4)  # 绘制UMAP图

In [4]:
# 加载测试数据（待注释的数据）
test_adata = sc.read_h5ad("../data/annotation_pancreas/demo_test.h5ad")

# 获取测试数据的细胞嵌入
test_embed_adata = scg.tasks.embed_data(
    test_adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=cell_type_key,  # 可选参数，用于保存元信息
    batch_size=64,
    return_new_adata=True,
)

# # 如果在CPU上运行
# test_embed_adata = scg.tasks.embed_data(
#     test_adata,
#     model_dir,
#     gene_col=gene_col,
#     obs_to_save=cell_type_key,
#     batch_size=64,
#     device="cpu",
#     use_fast_transformer=False,
#     return_new_adata=True,
# )

# # 可选步骤：可视化测试数据集
# sc.pp.neighbors(test_embed_adata, use_rep="X")
# sc.tl.umap(test_embed_adata)
# sc.pl.umap(test_embed_adata, color=cell_type_key, frameon=False, wspace=0.4)

In [5]:
# 当faiss不可用时使用的替代函数
def l2_sim(a, b):
    """计算L2相似度（取负值，使相似度越高值越大）"""
    sims = -np.linalg.norm(a - b, axis=1)
    return sims

def get_similar_vectors(vector, ref, top_k=10):
    """获取最相似的top_k个向量"""
    sims = l2_sim(vector, ref)
    top_k_idx = np.argsort(sims)[::-1][:top_k]  # 按相似度降序排列
    return top_k_idx, sims[top_k_idx]

In [6]:
# 提取细胞嵌入
ref_cell_embeddings = ref_embed_adata.X  # 参考数据的嵌入
test_emebd = test_embed_adata.X  # 测试数据的嵌入

k = 10  # 选择最近邻居的数量

# 使用faiss进行快速相似度搜索
if faiss_imported:
    # 创建L2距离索引
    index = faiss.IndexFlatL2(ref_cell_embeddings.shape[1])
    index.add(ref_cell_embeddings)  # 添加参考嵌入到索引

    # 查询测试数据，返回距离和标签
    distances, labels = index.search(test_emebd, k)

# 预测细胞类型
idx_list = [i for i in range(test_emebd.shape[0])]
preds = []
for idx in idx_list:
    if faiss_imported:
        nearest_idx = labels[idx]
    else:
        nearest_idx, sim = get_similar_vectors(test_emebd[idx][np.newaxis, ...], ref_cell_embeddings, k)
    
    # 通过多数投票确定预测的细胞类型
    pred = ref_embed_adata.obs[cell_type_key][nearest_idx].value_counts()
    preds.append(pred.index[0])

# 获取真实标签用于评估
gt = test_adata.obs[cell_type_key].to_numpy()

# 计算准确率
accuracy = sklearn.metrics.accuracy_score(gt, preds)
print(f"参考映射准确率: {accuracy:.4f}")

In [7]:
# # 可选：保存/加载构建的索引供将来使用
# faiss.write_index(index, "index.faiss")
# index = faiss.read_index("index.faiss")

## 方法二：使用CellXGene图谱进行参考映射

我们之前已经为正常或癌症样本中的所有细胞构建了索引，总共超过3300万个细胞。您可以在 [build_atlas_index_faiss.py](build_atlas_index_faiss.py) 找到构建索引的代码。
我们进行了仔细的调整，最终很好地平衡了准确性和效率。现在实际构建过程不到3分钟，我们选择只为每个细胞存储16字节的向量，这使得所有数百万细胞的整个索引只有808 MB。请从 https://drive.google.com/drive/folders/1q14U50SNg5LMjlZ9KH-n-YsGRi8zkCbe?usp=sharing 下载faiss索引文件夹。

使用此索引需要安装Faiss。请按照 https://github.com/facebookresearch/faiss/wiki/Installing-Faiss 的说明进行安装

In [8]:
# 导入索引加载和投票函数
from build_atlas_index_faiss import load_index, vote

In [9]:
# 检查是否有可用的GPU
use_gpu = faiss.get_num_gpus() > 0

# 加载CellXGene索引
index, meta_labels = load_index(
    index_dir="path_to_faiss_index_folder",  # 替换为您的索引路径
    use_config_file=False,
    use_gpu=use_gpu,
)

print(f"✅ 已加载索引，包含 {index.ntotal} 个细胞")

搜索速度非常快，尤其是在GPU上。在包含数百万细胞的整个参考数据集中搜索4000个查询细胞在CPU上大约需要7秒，在GPU上大约需要0.1秒。

In [10]:
# 测试搜索性能
k = 50  # 选择50个最近邻居

# 在测试数据上进行搜索
distances, idx = index.search(test_emebd, k)

print(f"✅ 搜索完成，查询了 {test_emebd.shape[0]} 个细胞")

在这里，我们通过多数投票从CellXGene注释中传播之前保存的细胞类型标签`meta_label`。

In [11]:
# 获取每个查询细胞的预测标签
predict_labels = meta_labels[idx]

from tqdm import tqdm

# 通过多数投票确定最终标签
voting = []
for preds in tqdm(predict_labels, desc="投票进行中"):
    voting.append(vote(preds, return_prob=False)[0])
voting = np.array(voting)

print("✅ 投票完成！")

In [12]:
# 输出前10个预测结果进行对比
print("真实标签（前10个）:", gt[:10])
print("预测标签（前10个）:", voting[:10])

目前，我们发现CellXGene标签分布在细胞类型层次结构的不同级别，其中更通用的细胞类型（如基质细胞）数量很大。每种细胞类型的细胞数量差异可能导致注释不太有用，尽管相似度搜索本身可能足够准确。我们正在努力以一致的方式更好地管理细胞类型标签。我们目前的想法是应用分层预测，并为每个细胞提供一系列从通用到更具体的细胞类型标签。

您可以通过运行以下代码查看元标签中细胞类型的比例：
```python
from build_atlas_index_faiss import compute_category_proportion
compute_category_proportion(meta_labels)
```

同时，主要细胞类型的传播通常更准确。下面提供了一个内皮细胞的例子。

In [13]:
# 查找内皮细胞
ids_m = np.where(gt == "endothelial")[0]
print(f"找到 {len(ids_m)} 个内皮细胞")
print(f"预测的细胞类型: {voting[ids_m]}")
print(f"注释的细胞类型: {gt[ids_m]}")